# Pure Component Property Calculator

This notebook demonstrates how to use the `PropertyCalculator` class to calculate pure component properties using the Constantinou-Gani group contribution method.

## Overview

The PropertyCalculator can calculate the following properties:
- **Tm**: Melting point [K]
- **Tb**: Boiling point [K] 
- **Tc**: Critical temperature [K]
- **Pc**: Critical pressure [bar]
- **Ac**: Acentric factor [-]
- **Hv**: Heat of vaporization [kJ/mol]

## References
- Constantinou, L., & Gani, R. (1994). New group contribution method for estimating properties of pure compounds. *AIChE Journal*, 40(10), 1697-1710.
- Constantinou, L., Gani, R., & O'Connell, J. P. (1995). Estimation of the acentric factor and the liquid molar volume at 298 K using a new group contribution method. *Fluid Phase Equilibria*, 103(1), 11-22.

## Setup and Imports

In [1]:
# Import necessary libraries
import sys
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Add src to path for imports - fix the path resolution
current_dir = Path(os.getcwd())
if current_dir.name == 'tutorials':
    # If running from tutorials directory
    src_path = current_dir.parent / "src"
else:
    # If running from project root
    src_path = current_dir / "src"

sys.path.insert(0, str(src_path))
print(f"Added to path: {src_path}")

# Now import the PropertyCalculator
try:
    from property_calculator import PropertyCalculator, calculate_properties
    print("✓ Successfully imported PropertyCalculator")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Python path: {sys.path[:3]}...")

# Initialize the calculator
try:
    calculator = PropertyCalculator()
    print("\n=== PropertyCalculator Initialized ===")
    print(f"Method: {calculator.pure_method}")
    print(f"Data path: {calculator.data_path}")
    print(f"Available properties: {calculator.get_available_properties()}")
except Exception as e:
    print(f"✗ Failed to initialize PropertyCalculator: {e}")

Added to path: /Users/arnaou/projects/dtu_chemeng/camd_iss/src
✓ Successfully imported PropertyCalculator
Loading property data from: /Users/arnaou/projects/dtu_chemeng/camd_iss/data/json
  Loaded scalars: 9 entries
  Loaded tm: 119 entries
  Loaded tb: 119 entries
  Loaded tc: 119 entries
  Loaded pc: 119 entries
  Loaded ac: 119 entries
  Loaded hv: 119 entries

=== PropertyCalculator Initialized ===
Method: Constantinou-Gani
Data path: /Users/arnaou/projects/dtu_chemeng/camd_iss/data/json
Available properties: ['tm', 'tb', 'tc', 'pc', 'ac', 'hv']


## Example 1: Single Compound Calculation

Let's calculate properties for ethanol (C₂H₅OH) with group composition: CH₃-CH₂-OH

In [2]:
# Define ethanol composition
ethanol = {'CH3': 1, 'CH2': 1, 'OH': 1}

print("ETHANOL Group Composition:", ethanol)
print("\n--- Individual Property Calculations ---")

# Calculate individual properties
tm = calculator.calculate_tm(ethanol)
tb = calculator.calculate_tb(ethanol)
tc = calculator.calculate_tc(ethanol)
pc = calculator.calculate_pc(ethanol)
ac = calculator.calculate_ac(ethanol)
hv = calculator.calculate_hv(ethanol)

# Display results
if tm: print(f"Melting point (Tm):      {tm:.2f} K")
if tb: print(f"Boiling point (Tb):     {tb:.2f} K")
if tc: print(f"Critical temperature:   {tc:.2f} K")
if pc: print(f"Critical pressure:      {pc:.2f} bar")
if ac: print(f"Acentric factor:        {ac:.3f} [-]")
if hv: print(f"Heat of vaporization:   {hv:.2f} kJ/mol")

# Compare with experimental values (approximate)
print("\n--- Comparison with Experimental Values ---")
exp_values = {
    'Tm': 159.0,   # K
    'Tb': 351.4,   # K  
    'Tc': 514.0,   # K
    'Pc': 61.4,    # bar
    'Ac': 0.64,    # [-]
    'Hv': 42.3     # kJ/mol
}

calc_values = {'tm': tm, 'tb': tb, 'tc': tc, 'pc': pc, 'ac': ac, 'hv': hv}

for prop in exp_values:
    if calc_values[prop.lower()]:
        calc_val = calc_values[prop.lower()]
        exp_val = exp_values[prop]
        error = abs(calc_val - exp_val) / exp_val * 100
        print(f"{prop}: Calculated = {calc_val:.2f}, Experimental = {exp_val:.2f}, Error = {error:.1f}%")

ETHANOL Group Composition: {'CH3': 1, 'CH2': 1, 'OH': 1}

--- Individual Property Calculations ---
Melting point (Tm):      164.57 K
Boiling point (Tb):     330.01 K
Critical temperature:   489.28 K
Critical pressure:      55.57 bar
Acentric factor:        0.527 [-]
Heat of vaporization:   40.12 kJ/mol

--- Comparison with Experimental Values ---
Tm: Calculated = 164.57, Experimental = 159.00, Error = 3.5%
Tb: Calculated = 330.01, Experimental = 351.40, Error = 6.1%
Tc: Calculated = 489.28, Experimental = 514.00, Error = 4.8%
Pc: Calculated = 55.57, Experimental = 61.40, Error = 9.5%
Ac: Calculated = 0.53, Experimental = 0.64, Error = 17.7%
Hv: Calculated = 40.12, Experimental = 42.30, Error = 5.1%


## Example 2: Calculate All Properties at Once

Using the `calculate_all_properties` method for convenience

In [3]:
# Calculate all properties for ethanol
properties = calculator.calculate_all_properties(ethanol)

print("ETHANOL - All Properties:")
print("=" * 40)

units = {'tm': 'K', 'tb': 'K', 'tc': 'K', 'pc': 'bar', 'ac': '[-]', 'hv': 'kJ/mol'}

if properties:
    for prop_name, value in properties.items():
        if value is not None:
            unit = units.get(prop_name, '')
            print(f"{prop_name.upper():3s}: {value:8.2f} {unit}")
        else:
            print(f"{prop_name.upper():3s}: Not available")
else:
    print("Could not calculate properties (missing group data)")

ETHANOL - All Properties:
TM :   164.57 K
TB :   330.01 K
TC :   489.28 K
PC :    55.57 bar
AC :     0.53 [-]
HV :    40.12 kJ/mol


## Example 3: Multiple Compounds Comparison

Calculate properties for several different compounds

In [4]:
# Define multiple test compounds
test_compounds = {
    'Water': {'H2O': 1},
    'Methanol': {'CH3OH': 1},
    'Ethanol': {'CH3': 1, 'CH2': 1, 'OH': 1},
    'Acetone': {'CH3': 1, 'CH3CO': 1},
    'Heptane': {'CH3': 2, 'CH2': 5},
    'Benzene': {'ACH': 6}
}

print("=== Multiple Compounds Calculation ===")
print()

results = {}

for name, compound in test_compounds.items():
    print(f"{name.upper()} ({compound}):")
    properties = calculator.calculate_all_properties(compound)
    
    if properties:
        results[name] = properties
        for prop_name, value in properties.items():
            if value is not None:
                unit = units.get(prop_name, '')
                print(f"  {prop_name.upper()}: {value:8.2f} {unit}")
            else:
                print(f"  {prop_name.upper()}: Not available")
    else:
        print("  Could not calculate properties")
    print()

=== Multiple Compounds Calculation ===

WATER ({'H2O': 1}):
  TM:   273.15 K
  TB:   373.15 K
  TC:   647.00 K
  PC:   220.64 bar
  AC:     0.34 [-]
  HV:    43.99 kJ/mol

METHANOL ({'CH3OH': 1}):
  TM:   176.00 K
  TB:   337.85 K
  TC:   513.00 K
  PC:    81.00 bar
  AC:     0.56 [-]
  HV:    37.47 kJ/mol

ETHANOL ({'CH3': 1, 'CH2': 1, 'OH': 1}):
  TM:   164.57 K
  TB:   330.01 K
  TC:   489.28 K
  PC:    55.57 bar
  AC:     0.53 [-]
  HV:    40.12 kJ/mol

ACETONE ({'CH3': 1, 'CH3CO': 1}):
  TM:   171.62 K
  TB:   305.37 K
  TC:   490.11 K
  PC:    48.80 bar
  AC:     0.33 [-]
  HV:    29.94 kJ/mol

HEPTANE ({'CH3': 2, 'CH2': 5}):
  TM:   175.55 K
  TB:   379.07 K
  TC:   549.86 K
  PC:    28.27 bar
  AC:     0.34 [-]
  HV:    38.31 kJ/mol

BENZENE ({'ACH': 6}):
  TM:   222.77 K
  TB:   351.27 K
  TC:   563.16 K
  PC:    48.62 bar
  AC:     0.22 [-]
  HV:    31.42 kJ/mol



## Example 4: Create a Summary Table

Organize results into a pandas DataFrame for better visualization

In [8]:
# Create DataFrame from results
if results:
    df_results = pd.DataFrame(results).T
    
    print("=== Properties Summary Table ===")
    print()
    
    # Round values for better display
    df_display = df_results.round(2)
    
    # Add units to column names
    column_mapping = {
        'tm': 'Tm [K]',
        'tb': 'Tb [K]', 
        'tc': 'Tc [K]',
        'pc': 'Pc [bar]',
        'ac': 'Ac [-]',
        'hv': 'Hv [kJ/mol]'
    }
    
    df_display = df_display.rename(columns=column_mapping)
    
    print(df_display)
    
    # Basic statistics
    print("\n=== Summary Statistics ===")
    print(f"Number of compounds calculated: {len(df_results)}")
    print(f"Properties calculated: {list(df_results.columns)}")
    
    # Count non-null values per property
    print("\nData availability:")
    for col in df_results.columns:
        non_null = df_results[col].notna().sum()
        print(f"  {col.upper()}: {non_null}/{len(df_results)} compounds")

=== Properties Summary Table ===

          Tm [K]  Tb [K]  Tc [K]  Pc [bar]  Ac [-]  Hv [kJ/mol]
Water     273.15  373.15  647.00    220.64    0.34        43.99
Methanol  176.00  337.85  513.00     81.00    0.56        37.47
Ethanol   164.57  330.01  489.28     55.57    0.53        40.12
Acetone   171.62  305.37  490.11     48.80    0.33        29.94
Heptane   175.55  379.07  549.86     28.27    0.34        38.31
Benzene   222.77  351.27  563.16     48.62    0.22        31.42

=== Summary Statistics ===
Number of compounds calculated: 6
Properties calculated: ['tm', 'tb', 'tc', 'pc', 'ac', 'hv']

Data availability:
  TM: 6/6 compounds
  TB: 6/6 compounds
  TC: 6/6 compounds
  PC: 6/6 compounds
  AC: 6/6 compounds
  HV: 6/6 compounds


## Example 5: Using the Convenience Function

For quick calculations, you can use the standalone `calculate_properties` function

In [6]:
# Quick calculation using convenience function
compound = {'CH3': 2, 'CH3CO': 1}  # Acetone

print("=== Quick Calculation (Acetone) ===")
print(f"Compound: {compound}")
print()

props = calculate_properties(compound)

if props:
    for prop_name, value in props.items():
        if value is not None:
            unit = units.get(prop_name, '')
            print(f"{prop_name.upper()}: {value:.2f} {unit}")
        else:
            print(f"{prop_name.upper()}: Not available")
else:
    print("Could not calculate properties")

=== Quick Calculation (Acetone) ===
Compound: {'CH3': 2, 'CH3CO': 1}

Loading property data from: /Users/arnaou/projects/dtu_chemeng/camd_iss/data/json
  Loaded scalars: 9 entries
  Loaded tm: 119 entries
  Loaded tb: 119 entries
  Loaded tc: 119 entries
  Loaded pc: 119 entries
  Loaded ac: 119 entries
  Loaded hv: 119 entries
TM: 180.15 K
TB: 342.56 K
TC: 509.36 K
PC: 38.06 bar
AC: 0.42 [-]
HV: 34.06 kJ/mol


## Example 6: Handling Missing Groups

Demonstration of how the calculator handles unknown or missing functional groups

In [7]:
print("=== Missing Groups Handling ===")
print()

# Example with unknown group
unknown_compound = {'CH3': 1, 'UNKNOWN_GROUP': 1}

print(f"Compound with unknown group: {unknown_compound}")

# Default behavior (remove compounds)
calc_remove = PropertyCalculator(missing_group_action="remove_compounds")
props_remove = calc_remove.calculate_all_properties(unknown_compound)

print(f"With 'remove_compounds' action: {props_remove}")

# Show missing groups summary
print("\nMissing groups summary:")
calc_remove.print_missing_groups_summary()

print("\n" + "="*50)
print("This demonstrates how the calculator handles missing group contributions.")
print("The default behavior is to remove compounds with missing data to ensure")
print("reliable property predictions.")

=== Missing Groups Handling ===

Compound with unknown group: {'CH3': 1, 'UNKNOWN_GROUP': 1}
Loading property data from: /Users/arnaou/projects/dtu_chemeng/camd_iss/data/json
  Loaded scalars: 9 entries
  Loaded tm: 119 entries
  Loaded tb: 119 entries
  Loaded tc: 119 entries
  Loaded pc: 119 entries
  Loaded ac: 119 entries
  Loaded hv: 119 entries
         Action: Removing compound (missing_group_action='remove_compounds')
With 'remove_compounds' action: None

Missing groups summary:

=== Missing Group Contributions Summary ===
TM: ['UNKNOWN_GROUP']
Action taken: remove_compounds

This demonstrates how the calculator handles missing group contributions.
The default behavior is to remove compounds with missing data to ensure
reliable property predictions.


## Conclusion

The `PropertyCalculator` class provides a robust implementation of the Constantinou-Gani group contribution method for estimating pure component properties. Key features include:

1. **Easy to use**: Simple dictionary input for molecular group compositions
2. **Comprehensive**: Calculates 6 important thermodynamic properties
3. **Reliable**: Handles missing data gracefully with configurable actions
4. **Flexible**: Individual property calculation or all-at-once calculation
5. **Informative**: Provides detailed logging and missing group summaries

This tool is particularly useful for:
- Process design and simulation
- Solvent screening and selection
- Computer-aided molecular design (CAMD)
- Property estimation for novel compounds

## Next Steps

- Explore integration with UNIFAC for mixture properties
- Use in computer-aided molecular design workflows
- Combine with experimental data for validation studies